# 03 — Uncertainty analysis: parametric propagation, MC vs RQMC, regimes and measurement error

**Objective.** Quantify, with the verified solver, how uncertainty enters reactor predictions, keeping four
error sources strictly separate:

| Source | Model | Propagation | Section |
|---|---|---|---|
| parametric uncertainty | $u,D,k,C_\mathrm{in}$ independent log-normal (assumed, illustrative) | forward, through the PDE | 1–3, 5 |
| sampling (integration) error | MC vs scrambled-Sobol' RQMC estimators of expectations | replicate spread | 2 |
| measurement error | $y=C_\mathrm{true}+b+\varepsilon$, fixed offset $b$, $\varepsilon\sim\mathcal N(0,\sigma^2)$ i.i.d. | inverse (parameter inference) | 6 |
| numerical error | discretisation bias of the solver | grid/step refinement (notebook 02) | 7 |

**Runtime.** About 20–25 minutes on a laptop (dominated by $\sim3\times10^5$ batched transient PDE solves).
All random numbers derive from the documented master seed below through `numpy.random.SeedSequence.spawn`.

In [1]:
# --- Environment set-up (no hidden state: every notebook starts here) -------
import sys
from pathlib import Path

try:
    import pfr_uq  # installed with `pip install -e .`
except ImportError:  # fall back to the repository's src/ directory
    root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
    sys.path.insert(0, str(root / "src"))
    import pfr_uq

import numpy as np
import pandas as pd
from pfr_uq import model, solvers, metrics, uncertainty, reporting, paths

np.set_printoptions(precision=6, suppress=False)
print("pfr_uq", pfr_uq.__version__, "| numpy", np.__version__, "| repository root:", paths.ROOT.name)

pfr_uq 1.0.0 | numpy 2.5.3 | repository root: Mohamed_AIMS_Project


In [2]:
import time
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from scipy.stats import norm as normal
from pfr_uq import qoi

MASTER_SEED = 20261002
SEED_INDEX = {"mc_replicates": 0, "rqmc_replicates": 1, "steady_mc": 2, "steady_rqmc": 3, "sobol_baseline": 4,
              "regime_map": 5, "steady_inference": 6, "transient_fit": 7, "bootstrap": 8, "example_noise": 9}
def child_rngs(name, n):
    """n independent generators for experiment `name`; identical on every call (no dependence on cell order)."""
    ss = np.random.SeedSequence(entropy=MASTER_SEED, spawn_key=(SEED_INDEX[name],))
    return [np.random.default_rng(s) for s in ss.spawn(n)]

SUMMARY = {"master_seed": MASTER_SEED, "seed_index": SEED_INDEX}
P = model.BASELINE
QN = list(qoi.QOI_NAMES)          # ["C_out", "X", "peak", "t_mean", "exposure"]
print("QoIs:", QN, "| production settings:", qoi.PULSE)

QoIs: ['C_out', 'X', 'peak', 't_mean', 'exposure'] | production settings: PulseSettings(t_center=5.0, width=1.0, t_end=80.0, dt=0.1, N=100, L=1.0)


## 1. Input uncertainty model (assumed)

Each positive parameter is log-normal with median equal to the baseline value and coefficient of variation (CV)
chosen for illustration: velocity 10 % (flow metering), dispersion coefficient 30 % (dispersion correlations are
known to scatter widely), rate constant 20 % (kinetic fit), feed concentration 5 %. These distributions are
**assumptions**, not measurements; the methodology, not the specific numbers, is the contribution.
Log-normal variables are parameterised so that the CV is exact: $\ln X\sim\mathcal N(\ln m,s^2)$, $s^2=\ln(1+\mathrm{CV}^2)$.

In [3]:
INPUT_SPEC = {"u": (P.u, 0.10, "m s^-1"), "D": (P.D, 0.30, "m^2 s^-1"), "k": (P.k, 0.20, "s^-1"), "C_in": (P.C_in, 0.05, "mol m^-3")}
inputs = uncertainty.InputModel.from_dict({name: uncertainty.LogNormal(m, cv) for name, (m, cv, _) in INPUT_SPEC.items()})
rows = []
for name, marg in zip(inputs.names, inputs.marginals):
    lo, hi = marg.quantile_interval(0.95)
    rows.append([name, INPUT_SPEC[name][2], marg.median, marg.cv, marg.mean, marg.sd, lo, hi])
input_table = pd.DataFrame(rows, columns=["input", "unit", "median", "CV", "mean", "sd", "2.5%", "97.5%"])
input_table.to_csv(paths.TABLES / "input_distributions.csv", index=False)
SUMMARY["inputs"] = input_table.to_dict(orient="records")
print(input_table.to_string(index=False, float_format=lambda v: f"{v:.4g}"))

CHUNK = 4096
def evaluate(U):
    """All five QoIs for points U in [0,1)^4 (columns ordered as QN); batched PDE solves in chunks."""
    out, min_c = [], np.inf
    for start in range(0, U.shape[0], CHUNK):
        p = inputs.transform(U[start:start + CHUNK])
        s = qoi.steady_qois(p["u"], p["D"], p["k"], p["C_in"])
        q = qoi.pulse_qois(p["u"], p["D"], p["k"], p["C_in"])
        min_c = min(min_c, q["min_value"])
        out.append(np.column_stack([s["C_out"], s["X"], q["peak"], q["t_mean"], q["exposure"]]))
    return np.vstack(out), min_c

input     unit  median   CV    mean       sd     2.5%   97.5%
    u   m s^-1     0.1  0.1  0.1005  0.01005  0.08224  0.1216
    D m^2 s^-1    0.01  0.3 0.01044 0.003132 0.005625 0.01778
    k     s^-1     0.1  0.2   0.102   0.0204  0.06783  0.1474
 C_in mol m^-3       1 0.05   1.001  0.05006   0.9067   1.103


## 2. Monte Carlo versus randomized quasi-Monte Carlo

Both methods estimate $\mathbb E[Q]$ and $\operatorname{Var}[Q]$ for the *same* input distribution: points in
$[0,1)^4$ are mapped by the same inverse-CDF transform. MC uses independent PCG64 streams; RQMC uses Sobol' points
with linear matrix scrambling and digital shift (`scipy.stats.qmc.Sobol(scramble=True)`), sample sizes
$n=2^m$, $m=4,\dots,12$ (nested prefixes of one $2^{12}$-point set, each a complete net).
$R=32$ independent replicates per method. Both mean estimators are unbiased, so their root-mean-square error is
estimated by the replicate standard deviation. For the variance we use, for each method, an estimator whose bias is
negligible: under MC the usual $s^2$ with divisor $n-1$ is unbiased, whereas under RQMC
$\operatorname{Var}(\bar Q_n)\ll\sigma^2/n$, so the divisor $n-1$ would introduce a bias $\approx\sigma^2/(n-1)$; RQMC therefore
uses the divisor $n$ (bias $-\operatorname{Var}(\bar Q_n)$, negligible). The RMSE of the variance estimators is measured about
the pooled RQMC estimate at $n=2^{12}$ (whose own standard error is reported).

In [4]:
R, M_MAX = 32, 12
SIZES = [2**m for m in range(4, M_MAX + 1)]
replicates = {}
pooled_Q = []
min_conc = np.inf
t0 = time.time()
for method, key in (("mc", "mc_replicates"), ("rqmc", "rqmc_replicates")):
    rngs = child_rngs(key, R)
    means, variances = [], []
    for r in range(R):
        U = uncertainty.unit_cube_sample(2**M_MAX, inputs.dim, method, rngs[r])
        Q, mn = evaluate(U)
        min_conc = min(min_conc, mn)
        means.append(uncertainty.nested_estimates(Q, SIZES, np.mean))
        ddof = 1 if method == "mc" else 0   # unbiased for MC; negligible bias for RQMC (see text above)
        variances.append(uncertainty.nested_estimates(Q, SIZES, lambda a, axis, dd=ddof: np.var(a, axis=axis, ddof=dd)))
        if method == "rqmc":
            pooled_Q.append(Q)
    replicates[method] = {"mean": np.array(means), "var": np.array(variances)}   # (R, len(SIZES), 5)
print(f"elapsed {time.time() - t0:.0f} s; minimum concentration over all transient runs: {min_conc:.3e}")
pooled_Q = np.vstack(pooled_Q)   # 32 * 4096 RQMC samples (scrambles independent)
np.savez_compressed(paths.DATA / "mcqmc_replicates.npz", sizes=np.array(SIZES), qoi_names=np.array(QN),
                    mc_mean=replicates["mc"]["mean"], mc_var=replicates["mc"]["var"],
                    rqmc_mean=replicates["rqmc"]["mean"], rqmc_var=replicates["rqmc"]["var"])
SUMMARY["min_concentration_uq_runs"] = float(min_conc)

elapsed 641 s; minimum concentration over all transient runs: 0.000e+00


In [5]:
# Reference values: pooled RQMC estimates at n = 2^12 over the 32 replicates (standard error from replicate spread)
ref_mean = replicates["rqmc"]["mean"][:, -1].mean(axis=0)
ref_mean_se = replicates["rqmc"]["mean"][:, -1].std(axis=0, ddof=1) / np.sqrt(R)
ref_var = replicates["rqmc"]["var"][:, -1].mean(axis=0)
ref_var_se = replicates["rqmc"]["var"][:, -1].std(axis=0, ddof=1) / np.sqrt(R)

def rmse_curves(method, stat):
    est = replicates[method][stat]                     # (R, S, 5)
    if stat == "mean":
        return est.std(axis=0, ddof=1)                 # replicate sd (unbiased estimators)
    return np.sqrt(np.mean((est - ref_var) ** 2, axis=0))

boot_rng = child_rngs("bootstrap", 1)[0]
fit_mask = np.array(SIZES) >= 2**6
rows, curves = [], []
for stat in ("mean", "var"):
    for method in ("mc", "rqmc"):
        err = rmse_curves(method, stat)
        for j, name in enumerate(QN):
            slope, se, _ = metrics.fitted_order(np.array(SIZES)[fit_mask], err[fit_mask, j])
            def slope_stat(sample, j=j, stat=stat):
                e = sample.std(axis=0, ddof=1) if stat == "mean" else np.sqrt(np.mean((sample - ref_var) ** 2, axis=0))
                return metrics.fitted_order(np.array(SIZES)[fit_mask], e[fit_mask, j])[0]
            lo, hi = uncertainty.bootstrap_ci(replicates[method][stat], slope_stat, 400, boot_rng)
            rows.append({"estimator": stat, "method": method, "QoI": name, "rate": slope, "rate 95% CI low": lo,
                         "rate 95% CI high": hi, "RMSE at n=4096": err[-1, j], "RMSE at n=16": err[0, j]})
            for s_i, n_s in enumerate(SIZES):
                curves.append({"estimator": stat, "method": method, "QoI": name, "n": n_s, "rmse": err[s_i, j]})
rates = pd.DataFrame(rows)
pd.DataFrame(curves).to_csv(paths.DATA / "mcqmc_rmse_curves.csv", index=False)
# variance-reduction factor (MSE_MC / MSE_RQMC) at n = 4096 with bootstrap CI over replicates
vrf_rows = []
for stat in ("mean", "var"):
    for j, name in enumerate(QN):
        e_mc = replicates["mc"][stat][:, -1, j]; e_q = replicates["rqmc"][stat][:, -1, j]
        if stat == "mean":
            f = lambda a, b: np.var(a, ddof=1) / np.var(b, ddof=1)
        else:
            f = lambda a, b: np.mean((a - ref_var[j]) ** 2) / np.mean((b - ref_var[j]) ** 2)
        point = f(e_mc, e_q)
        boots = []
        for _ in range(2000):
            i1 = boot_rng.integers(0, R, R); i2 = boot_rng.integers(0, R, R)
            boots.append(f(e_mc[i1], e_q[i2]))
        lo, hi = np.quantile(boots, [0.025, 0.975])
        vrf_rows.append({"estimator": stat, "QoI": name, "MSE ratio MC/RQMC": point, "95% CI low": lo, "95% CI high": hi})
vrf = pd.DataFrame(vrf_rows)
rates.to_csv(paths.TABLES / "mcqmc_rates.csv", index=False)
vrf.to_csv(paths.TABLES / "mcqmc_efficiency.csv", index=False)
print(rates.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print(vrf.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
SUMMARY["mcqmc"] = {"R": R, "sizes": SIZES, "rates": rates.to_dict(orient="records"), "efficiency": vrf.to_dict(orient="records"),
                    "reference_mean": dict(zip(QN, ref_mean)), "reference_mean_se": dict(zip(QN, ref_mean_se)),
                    "reference_var": dict(zip(QN, ref_var)), "reference_var_se": dict(zip(QN, ref_var_se))}

estimator method      QoI   rate  rate 95% CI low  rate 95% CI high  RMSE at n=4096  RMSE at n=16
     mean     mc    C_out -0.535            -0.65            -0.425         0.00117        0.0225
     mean     mc        X -0.538           -0.635            -0.439         0.00119        0.0218
     mean     mc     peak -0.503           -0.608            -0.402        0.000427       0.00808
     mean     mc   t_mean -0.445           -0.513            -0.359          0.0129          0.17
     mean     mc exposure -0.535           -0.642            -0.431         0.00293        0.0564
     mean   rqmc    C_out  -1.16            -1.24             -1.09        5.56e-06       0.00303
     mean   rqmc        X  -1.17            -1.27              -1.1        4.71e-06       0.00282
     mean   rqmc     peak  -1.11            -1.18             -1.04        2.89e-06       0.00139
     mean   rqmc   t_mean  -1.07            -1.18             -0.94         6.4e-05        0.0282
     mean   rqmc exp

**Interpretation.** Observed MC rates should be close to $n^{-1/2}$ (a theorem for finite-variance integrands);
any RQMC rate faster than that is an *empirical* property of these integrands, which are smooth functions of the
inputs. The peak concentration is obtained by parabolic interpolation around the discrete maximum, which makes it
only piecewise smooth in the inputs; this can reduce the RQMC advantage and is reported as observed.

### Extended study for the steady outlet concentration (closed form)

The steady exit concentration $C_\mathrm{in}\,G(\mathrm{Pe},\mathrm{Da})$ is available in closed form (verified against the
finite-volume solver in notebook 02), so the asymptotic behaviour can be examined up to $n=2^{18}$ with $R=64$.

In [6]:
def steady_closed_form(U):
    p = inputs.transform(U)
    return p["C_in"] * model.danckwerts_outlet_ratio(model.peclet(p["u"], P.L, p["D"]), model.damkohler(p["k"], P.L, p["u"]))

R_S, M_S = 64, 18
SIZES_S = [2**m for m in range(4, M_S + 1)]
ext = {}
for method, key in (("mc", "steady_mc"), ("rqmc", "steady_rqmc")):
    rngs = child_rngs(key, R_S)
    ext[method] = np.array([uncertainty.nested_estimates(steady_closed_form(
        uncertainty.unit_cube_sample(2**M_S, inputs.dim, method, rngs[r])), SIZES_S) for r in range(R_S)])
ext_err = {m: ext[m].std(axis=0, ddof=1) for m in ext}
mask_s = np.array(SIZES_S) >= 2**8
ext_rates = {m: metrics.fitted_order(np.array(SIZES_S)[mask_s], ext_err[m][mask_s]) for m in ext}
pd.DataFrame({"n": SIZES_S, "rmse_mc": ext_err["mc"], "rmse_rqmc": ext_err["rqmc"]}).to_csv(paths.DATA / "mcqmc_steady_extended.csv", index=False)
cout_mean_ext = ext["rqmc"][:, -1].mean()
print({m: (round(r[0], 3), round(r[1], 3)) for m, r in ext_rates.items()}, "| E[C_out] =", cout_mean_ext,
      "| RMSE ratio at 2^18:", ext_err["mc"][-1] / ext_err["rqmc"][-1])
SUMMARY["mcqmc_steady_extended"] = {"R": R_S, "max_n": 2**M_S, "rate_mc": ext_rates["mc"][0], "rate_mc_se": ext_rates["mc"][1],
                                    "rate_rqmc": ext_rates["rqmc"][0], "rate_rqmc_se": ext_rates["rqmc"][1],
                                    "rmse_mc_max_n": ext_err["mc"][-1], "rmse_rqmc_max_n": ext_err["rqmc"][-1],
                                    "E_Cout": cout_mean_ext, "E_Cout_FV_pooled": float(ref_mean[0])}

{'mc': (-0.49, 0.007), 'rqmc': (-1.081, 0.009)} | E[C_out] = 0.39857010057158676 | RMSE ratio at 2^18: 2121.9727165099735


## 3. Forward propagation at the baseline

Pooled statistics of the $32\times4096$ RQMC samples. Standard errors of the means are from the replicate spread;
the 95 % prediction interval is the 2.5–97.5 % quantile range of the pooled sample (quantile uncertainty from the
spread of replicate-wise quantiles). The deterministic prediction at the median inputs is shown for comparison.

In [7]:
nominal_s = qoi.steady_qois(P.u, P.D, P.k, P.C_in)
nominal_p = qoi.pulse_qois(P.u, P.D, P.k, P.C_in)
nominal = {"C_out": nominal_s["C_out"][0], "X": nominal_s["X"][0], "peak": nominal_p["peak"][0],
           "t_mean": nominal_p["t_mean"][0], "exposure": nominal_p["exposure"][0]}
q_rep = np.array([np.quantile(pooled_Q[r * 2**M_MAX:(r + 1) * 2**M_MAX], [0.025, 0.975], axis=0) for r in range(R)])  # (R, 2, 5)
rows = []
for j, name in enumerate(QN):
    col = pooled_Q[:, j]
    lo, hi = np.quantile(col, [0.025, 0.975])
    rows.append({"QoI": name, "unit": qoi.QOI_UNITS[name], "nominal": nominal[name], "mean": ref_mean[j], "mean SE": ref_mean_se[j],
                 "sd": np.sqrt(ref_var[j]), "CV": np.sqrt(ref_var[j]) / abs(ref_mean[j]), "2.5%": lo, "97.5%": hi,
                 "quantile SE (2.5%)": q_rep[:, 0, j].std(ddof=1) / np.sqrt(R), "quantile SE (97.5%)": q_rep[:, 1, j].std(ddof=1) / np.sqrt(R)})
forward = pd.DataFrame(rows)
forward.to_csv(paths.TABLES / "forward_uq_baseline.csv", index=False)
print(forward.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
# histograms for the figure
hist = {}
for j, name in enumerate(QN):
    counts, edges = np.histogram(pooled_Q[:, j], bins=60, density=True)
    hist[name] = {"density": counts, "edges": edges}
np.savez_compressed(paths.DATA / "forward_uq_histograms.npz", **{f"{n}_density": hist[n]["density"] for n in QN},
                    **{f"{n}_edges": hist[n]["edges"] for n in QN})
# exposure check against its closed form inside the UQ sample
U_chk = uncertainty.unit_cube_sample(4096, inputs.dim, "rqmc", child_rngs("rqmc_replicates", 1)[0])
p_chk = inputs.transform(U_chk)
exp_fv = evaluate(U_chk)[0][:, 4]
exp_cf = qoi.exposure_exact(p_chk["u"], p_chk["D"], p_chk["k"], p_chk["C_in"])
print("exposure: max relative difference FV vs closed form over 4096 samples:", np.max(np.abs(exp_fv / exp_cf - 1)))
SUMMARY["forward"] = forward.to_dict(orient="records")
SUMMARY["exposure_fv_vs_closed_form_max_rel"] = float(np.max(np.abs(exp_fv / exp_cf - 1)))

     QoI       unit  nominal   mean   mean SE      sd      CV    2.5%  97.5%  quantile SE (2.5%)  quantile SE (97.5%)
   C_out   mol m^-3   0.3973 0.3986 9.829e-07 0.07606  0.1908  0.2532 0.5505           0.0001935            0.0001989
       X          -   0.6027 0.6019 8.333e-07 0.07322  0.1216   0.459 0.7444           0.0001534            0.0001692
    peak   mol m^-3   0.1303 0.1329 5.102e-07  0.0296  0.2228 0.08151 0.1974           5.929e-05            0.0001563
  t_mean          s    13.57  13.55 1.132e-05  0.7446 0.05495   12.18   15.1            0.001936             0.002103
exposure mol s m^-3   0.9958 0.9991 2.464e-06  0.1907  0.1908  0.6347   1.38           0.0004851            0.0004985


exposure: max relative difference FV vs closed form over 4096 samples: 0.0001479001340682906


## 4. Variance-based sensitivity at the baseline (Sobol' indices)

First-order $S_i$ (Saltelli 2010 estimator) and total $S_{T,i}$ (Jansen estimator), base sample $n=2^{11}$ from a
scrambled Sobol' set in $[0,1)^8$, cost $n(d+2)=12\,288$ simulations per replicate, $R=8$ independent scrambles;
reported values are replicate means with standard errors.

In [8]:
R_SOB, N_SOB = 8, 2**11
def f_all(p):
    s = qoi.steady_qois(p["u"], p["D"], p["k"], p["C_in"])
    out = []
    for start in range(0, p["u"].size, CHUNK):
        sl = slice(start, start + CHUNK)
        q = qoi.pulse_qois(p["u"][sl], p["D"][sl], p["k"][sl], p["C_in"][sl])
        out.append(np.column_stack([q["peak"], q["t_mean"], q["exposure"]]))
    return np.column_stack([s["C_out"], s["X"], np.vstack(out)])
t0 = time.time()
sob = [uncertainty.sobol_indices(f_all, inputs, N_SOB, "rqmc", rng) for rng in child_rngs("sobol_baseline", R_SOB)]
print(f"Sobol' indices: {time.time() - t0:.0f} s")
S = np.array([s["S"] for s in sob]); ST = np.array([s["ST"] for s in sob])   # (R, d, q)
rows = []
for j, name in enumerate(QN):
    for i, inp in enumerate(inputs.names):
        rows.append({"QoI": name, "input": inp, "S": S[:, i, j].mean(), "S SE": S[:, i, j].std(ddof=1) / np.sqrt(R_SOB),
                     "ST": ST[:, i, j].mean(), "ST SE": ST[:, i, j].std(ddof=1) / np.sqrt(R_SOB)})
sobol_base = pd.DataFrame(rows)
sobol_base.to_csv(paths.TABLES / "sobol_baseline.csv", index=False)
print(sobol_base.pivot(index="input", columns="QoI", values="S").round(3).to_string())
print(sobol_base.pivot(index="input", columns="QoI", values="ST").round(3).to_string())
print("sum of first-order indices per QoI:", sobol_base.groupby("QoI")["S"].sum().round(3).to_dict())
SUMMARY["sobol_baseline"] = sobol_base.to_dict(orient="records")

Sobol' indices: 230 s
QoI    C_out      X  exposure   peak  t_mean
input                                       
C_in   0.068 -0.000     0.068  0.050   0.000
D      0.008  0.009     0.008  0.015   0.164
k      0.755  0.812     0.755  0.375   0.095
u      0.165  0.179     0.165  0.544   0.730
QoI    C_out      X  exposure   peak  t_mean
input                                       
C_in   0.071  0.000     0.071  0.053   0.000
D      0.009  0.009     0.009  0.023   0.173
k      0.757  0.813     0.757  0.384   0.102
u      0.166  0.178     0.166  0.554   0.736
sum of first-order indices per QoI: {'C_out': 0.996, 'X': 1.0, 'exposure': 0.996, 'peak': 0.984, 't_mean': 0.989}


## 5. Regime dependence: uncertainty across the Pe–Da plane

For nominal $(\mathrm{Pe}_0,\mathrm{Da}_0)$ on a logarithmic grid, $u$, $D$ and $k$ are multiplied by independent
log-normal factors with median 1 and the same CVs as above, so $\mathrm{Pe}=\mathrm{Pe}_0\,f_u/f_D$ and
$\mathrm{Da}=\mathrm{Da}_0\,f_k/f_u$. We map the CV of the steady outlet ratio $G=C_\mathrm{out}/C_\mathrm{in}$, the standard
deviation of the conversion $X=1-G$, and first-order Sobol' indices of $u,D,k$. The closed-form steady solution is
used (2 501 grid points × 20 480 evaluations); it is cross-checked against the finite-volume solver at selected points.

In [9]:
factors = uncertainty.InputModel.from_dict({n: uncertainty.LogNormal(1.0, INPUT_SPEC[n][1]) for n in ("u", "D", "k")})
PE_GRID = np.logspace(0, 3, 61)
DA_GRID = np.logspace(-1, 1, 41)
N_MAP = 2**12
rng_map = child_rngs("regime_map", 1)[0]
U_map = uncertainty.unit_cube_sample(N_MAP, 6, "rqmc", rng_map)   # common point set for all grid points
A_, B_ = factors.transform(U_map[:, :3]), factors.transform(U_map[:, 3:])
def hybrid(i):
    out = dict(A_); name = factors.names[i]; out[name] = B_[name]; return out
mats = [A_, B_] + [hybrid(i) for i in range(3)]
cv_G = np.empty((DA_GRID.size, PE_GRID.size)); sd_X = np.empty_like(cv_G); mean_X = np.empty_like(cv_G)
S_map = np.empty((3,) + cv_G.shape); ST_map = np.empty_like(S_map)
for a, Da0 in enumerate(DA_GRID):
    for b, Pe0 in enumerate(PE_GRID):
        vals = [model.danckwerts_outlet_ratio(Pe0 * m["u"] / m["D"], Da0 * m["k"] / m["u"]) for m in mats]
        fA, fB = vals[0], vals[1]
        V = np.var(np.concatenate([fA, fB]), ddof=1)
        allv = np.concatenate([fA, fB])
        cv_G[a, b] = allv.std(ddof=1) / allv.mean()
        sd_X[a, b] = allv.std(ddof=1)
        mean_X[a, b] = 1 - allv.mean()
        for i in range(3):
            S_map[i, a, b] = np.mean(fB * (vals[2 + i] - fA)) / V
            ST_map[i, a, b] = 0.5 * np.mean((fA - vals[2 + i]) ** 2) / V
np.savez_compressed(paths.DATA / "regime_maps.npz", Pe=PE_GRID, Da=DA_GRID, cv_G=cv_G, sd_X=sd_X, mean_X=mean_X,
                    S_u=S_map[0], S_D=S_map[1], S_k=S_map[2], ST_u=ST_map[0], ST_D=ST_map[1], ST_k=ST_map[2],
                    X_nominal=1 - model.danckwerts_outlet_ratio(PE_GRID[None, :], DA_GRID[:, None]),
                    X_pfr=1 - np.exp(-DA_GRID), X_cstr=DA_GRID / (1 + DA_GRID))
iD = np.unravel_index(np.argmax(S_map[1]), S_map[1].shape)
print(f"CV of G: min {cv_G.min():.3f}, max {cv_G.max():.3f}; sd of X: max {sd_X.max():.3f}")
print(f"max first-order index of D: {S_map[1].max():.3f} at Pe = {PE_GRID[iD[1]]:.3g}, Da = {DA_GRID[iD[0]]:.3g}")
print(f"index of D at Pe=1000 (max over Da): {S_map[1][:, -1].max():.4f}; at Pe=1 (max over Da): {S_map[1][:, 0].max():.3f}")
# selected regimes for the text
sel = []
for Pe0, Da0 in [(1, 1), (10, 1), (100, 1), (1000, 1), (10, 0.1), (10, 10), (1000, 10)]:
    a = np.argmin(abs(np.log(DA_GRID / Da0))); b = np.argmin(abs(np.log(PE_GRID / Pe0)))
    sel.append({"Pe": PE_GRID[b], "Da": DA_GRID[a], "X nominal": 1 - model.danckwerts_outlet_ratio(PE_GRID[b], DA_GRID[a]),
                "CV(G)": cv_G[a, b], "sd(X)": sd_X[a, b], "S_u": S_map[0, a, b], "S_D": S_map[1, a, b], "S_k": S_map[2, a, b],
                "ST_D": ST_map[1, a, b]})
regimes = pd.DataFrame(sel)
regimes.to_csv(paths.TABLES / "regime_selected.csv", index=False)
print(regimes.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
# cross-check of the closed form against the FV solver at the selected nominal points
chk = []
for r_ in sel:
    Pe0, Da0 = r_["Pe"], r_["Da"]
    N = int(max(200, 2 * Pe0))
    op = solvers.FVOperator(0.1, 0.1 / Pe0, Da0 * 0.1, 1.0, N)
    chk.append(abs(solvers.solve_steady(op, 1.0)[0, -1] / model.danckwerts_outlet_ratio(Pe0, Da0) - 1))
print("max relative difference FV vs closed form at selected points:", max(chk))
SUMMARY["regime"] = {"cv_G_min": cv_G.min(), "cv_G_max": cv_G.max(), "sd_X_max": sd_X.max(),
                     "S_D_max": S_map[1].max(), "S_D_max_Pe": PE_GRID[iD[1]], "S_D_max_Da": DA_GRID[iD[0]],
                     "S_D_max_at_Pe1000": S_map[1][:, -1].max(), "selected": regimes.to_dict(orient="records"),
                     "fv_check_max_rel": max(chk)}

CV of G: min 0.021, max 2.540; sd of X: max 0.080
max first-order index of D: 0.245 at Pe = 2, Da = 10
index of D at Pe=1000 (max over Da): 0.0001; at Pe=1 (max over Da): 0.226
   Pe  Da  X nominal  CV(G)    sd(X)    S_u      S_D   S_k     ST_D
    1   1      0.532   0.13   0.0609  0.171   0.0146 0.814    0.015
   10   1      0.603  0.184   0.0732  0.178  0.00889 0.813  0.00926
  100   1      0.629  0.212   0.0787  0.199 0.000193 0.801 0.000206
1e+03   1      0.632  0.216   0.0795  0.202 1.88e-06 0.798 2.26e-06
   10 0.1     0.0944 0.0223   0.0202  0.201 0.000131 0.801 0.000155
   10  10      0.998   1.07  0.00297 0.0646    0.131 0.694    0.184
1e+03  10          1   2.54 0.000577 0.0669 0.000111 0.591 0.000119
max relative difference FV vs closed form at selected points: 0.0007191380280568982


## 6. Measurement error: random noise versus systematic offset

### 6a. Steady-state inference of the rate constant

One analyser samples the inlet and outlet streams; each reading is $y=C+b+\varepsilon$ with
$\sigma=0.01$ mol m$^{-3}$ (1 % of the feed) and a fixed zero offset $b=0.01$ mol m$^{-3}$. With $n$ readings per port the
rate constant is inferred by inverting the steady Danckwerts solution at known $\mathrm{Pe}=10$:
$\hat k=(u/L)\,G^{-1}(\bar y_\mathrm{out}/\bar y_\mathrm{in})$. Three cases: (S1) $b=0$; (S2) $b\neq0$, uncorrected;
(S3) $b\neq0$, corrected with $n$ blank readings ($C=0$), $\hat b=\bar y_0$. Because the noise is Gaussian and i.i.d.,
the sample means are drawn exactly from $\mathcal N(C+b,\sigma^2/n)$. $R=20\,000$ replicates per $n$. Confidence intervals
use plug-in values ($\hat r$, $\bar y_\mathrm{in}$) and assume $\sigma$ known (in practice it would be estimated from the replicate
readings).

Delta-method predictions ($r=C_\mathrm{out}/C_\mathrm{in}$, $k'(r)=(u/L)\,dG^{-1}/dr$):
$\operatorname{Var}\hat r\approx\frac{\sigma^2}{C_\mathrm{in}^2}\left[\frac{1+r^2}{n}+\frac{(1-r)^2}{n_0}\mathbb 1_{S3}\right]$,
and for S2 the estimator converges to $k\!\left(\tfrac{C_\mathrm{out}+b}{C_\mathrm{in}+b}\right)\neq k$, a bias that does not
decrease with $n$.

In [10]:
sigma_m, b_m, R_M = 0.01, 0.01, 20000
C_in_true, r_true = P.C_in, float(model.danckwerts_outlet_ratio(P.Pe, P.Da))
C_out_true = r_true * C_in_true
k_true = P.k
def k_from_ratio(r):
    return P.u / P.L * model.damkohler_from_outlet_ratio(r, P.Pe)
def dk_dr(r, d=1e-6):
    return (k_from_ratio(r + d) - k_from_ratio(r - d)) / (2 * d)
rng_m = child_rngs("steady_inference", 1)[0]
N_READ = [2**j for j in range(0, 11)]
rows = []
for n in N_READ:
    z = rng_m.standard_normal((3, R_M)) * sigma_m / np.sqrt(n)
    ybar_in_0, ybar_out_0 = C_in_true + z[0], C_out_true + z[1]                 # S1 (no offset)
    ybar_in_b, ybar_out_b = C_in_true + b_m + z[0], C_out_true + b_m + z[1]     # S2/S3 (same noise draws)
    b_hat = b_m + z[2]                                                           # blank readings, n0 = n
    cases = {"S1 noise only": (ybar_out_0 / ybar_in_0, ybar_in_0, False),
             "S2 offset, uncorrected": (ybar_out_b / ybar_in_b, ybar_in_b, False),
             "S3 offset, blank-corrected": ((ybar_out_b - b_hat) / (ybar_in_b - b_hat), ybar_in_b - b_hat, True)}
    for case, (r_hat, cin_hat, blank) in cases.items():
        k_hat = k_from_ratio(r_hat)
        # plug-in delta-method standard error (r and C_in replaced by their estimates; sigma assumed known)
        var_fac_hat = (1 + r_hat**2) / n + ((1 - r_hat) ** 2 / n if blank else 0.0)
        se = np.abs(dk_dr(np.clip(r_hat, 1e-6, 1 - 1e-6))) * sigma_m / cin_hat * np.sqrt(var_fac_hat)
        var_fac = (1 + r_true**2) / n + ((1 - r_true) ** 2 / n if blank else 0.0)   # true-value prediction (column "sd delta-method")
        cover = np.abs(k_hat - k_true) <= normal.ppf(0.975) * se
        ok = np.isfinite(k_hat)
        lo_c, hi_c = uncertainty.wilson_interval(int(cover[ok].sum()), int(ok.sum()))
        rows.append({"case": case, "n": n, "bias": np.mean(k_hat[ok]) - k_true, "sd": np.std(k_hat[ok], ddof=1),
                     "rmse": np.sqrt(np.mean((k_hat[ok] - k_true) ** 2)),
                     "sd delta-method": abs(dk_dr(r_true)) * sigma_m / C_in_true * np.sqrt(var_fac),
                     "coverage 95% CI": cover[ok].mean(), "coverage CI low": lo_c, "coverage CI high": hi_c,
                     "invalid fraction": 1 - ok.mean()})
inference = pd.DataFrame(rows)
r_lim = (C_out_true + b_m) / (C_in_true + b_m)
bias_lim = k_from_ratio(r_lim) - k_true
sd1 = abs(dk_dr(r_true)) * sigma_m / C_in_true * np.sqrt(1 + r_true**2)
n_star = (sd1 / bias_lim) ** 2
inference.to_csv(paths.TABLES / "steady_inference.csv", index=False)
print(inference.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print(f"asymptotic offset bias k(r_b) - k = {bias_lim:.4g} s^-1 ({bias_lim / k_true:.2%}); crossover n* = {n_star:.3g}")
SUMMARY["steady_inference"] = {"sigma": sigma_m, "b": b_m, "R": R_M, "k_true": k_true, "r_true": r_true,
                               "asymptotic_bias": bias_lim, "asymptotic_rel_bias": bias_lim / k_true, "n_star": n_star,
                               "sd_single_reading": sd1, "table": inference.to_dict(orient="records")}

                      case    n       bias        sd      rmse  sd delta-method  coverage 95% CI  coverage CI low  coverage CI high  invalid fraction
             S1 noise only    1  1.642e-05  0.003149  0.003149          0.00316           0.9504           0.9473            0.9533                 0
    S2 offset, uncorrected    1  -0.001723  0.003071  0.003521          0.00316           0.9074           0.9034            0.9114                 0
S3 offset, blank-corrected    1  5.481e-05  0.003624  0.003624         0.003622           0.9517           0.9487            0.9546                 0
             S1 noise only    2  1.411e-05  0.002221  0.002221         0.002234           0.9517           0.9487            0.9546                 0
    S2 offset, uncorrected    2  -0.001724  0.002166  0.002769         0.002234           0.8683           0.8635            0.8729                 0
S3 offset, blank-corrected    2  2.892e-05  0.002551  0.002551         0.002561           0.9518    

### 6b. Calibrating $D$ and $k$ from a noisy pulse response (inverse propagation through the PDE)

Synthetic data: exit concentration of the baseline reactor after a Gaussian feed pulse ($t_c=5$ s, $w=1$ s,
amplitude 1 mol m$^{-3}$), sampled every 1 s on $[0,60]$ s (61 readings), generated with a **fine** grid
($N=800$, $\Delta t=0.0125$ s) to avoid an "inverse crime"; the fitting model uses the production grid ($N=100$,
$\Delta t=0.1$ s). Noise $\sigma=2\times10^{-3}$ mol m$^{-3}$ (≈1.5 % of the peak) and offset $b=2\times10^{-3}$ mol m$^{-3}$.
Two estimators are compared on the same 200 noise realisations: M1 fits $(\ln D,\ln k)$ ignoring the offset; M2 also
estimates $b$. Nonlinear least squares (Levenberg–Marquardt; finite-difference Jacobian from one batched solve);
95 % Wald intervals from $s^2(J^\top J)^{-1}$, $s^2=\mathrm{RSS}/(m-p)$.

In [11]:
T_OBS = np.arange(0.0, 60.0 + 1e-9, 1.0)
fit_set = qoi.PulseSettings(N=100, dt=0.1, t_end=60.0)
truth_set = qoi.PulseSettings(N=800, dt=0.0125, t_end=60.0)
def outlet_at_obs(D, k, settings):
    res = qoi.pulse_qois(P.u, D, k, 1.0, settings, return_curves=True)
    idx = np.rint(T_OBS / settings.dt).astype(int)
    return res["outlet"][idx]           # (m, B)
y_true = outlet_at_obs(P.D, P.k, truth_set)[:, 0]
sigma_t, b_t, R_T = 2e-3, 2e-3, 200
print(f"peak of true exit curve {y_true.max():.4f} mol m^-3; sigma / peak = {sigma_t / y_true.max():.3f}")

def make_problem(y, with_offset):
    cache = {}
    def sim(theta):
        key = tuple(np.round(theta, 14))
        if key not in cache:
            ths = [theta] + [theta + 1e-6 * np.eye(len(theta))[i] for i in range(2)]
            D_b = np.exp([t[0] for t in ths]); k_b = np.exp([t[1] for t in ths])
            cache.clear(); cache[key] = outlet_at_obs(D_b, k_b, fit_set)
        return cache[key]
    def fun(theta):
        out = sim(theta)[:, 0]
        return out + (theta[2] if with_offset else 0.0) - y
    def jac(theta):
        out = sim(theta)
        J = (out[:, 1:3] - out[:, [0]]) / 1e-6
        if with_offset:
            J = np.column_stack([J, np.ones(len(y))])
        return J
    return fun, jac

def fit(y, with_offset):
    fun, jac = make_problem(y, with_offset)
    x0 = np.array([np.log(0.015), np.log(0.08)] + ([0.0] if with_offset else []))
    sol = least_squares(fun, x0, jac=jac, method="lm", xtol=1e-12, ftol=1e-12)
    m, p = len(y), len(x0)
    s2 = np.sum(sol.fun**2) / (m - p)
    cov = s2 * np.linalg.inv(sol.jac.T @ sol.jac)
    return sol.x, np.sqrt(np.diag(cov)), sol

# numerical-model bias: fit the noise-free fine-grid truth with the production model
x_nf, _, _ = fit(y_true, False)
num_bias = {"D_rel": np.exp(x_nf[0]) / P.D - 1, "k_rel": np.exp(x_nf[1]) / P.k - 1}
print("noise-free fit (pure discretisation effect): relative errors", {k_: f"{v:.2e}" for k_, v in num_bias.items()})

rng_t = child_rngs("transient_fit", 1)[0]
t0 = time.time()
records = []
for rep in range(R_T):
    eps = sigma_t * rng_t.standard_normal(T_OBS.size)
    y = y_true + b_t + eps
    for label, off in (("M1 no offset", False), ("M2 offset estimated", True)):
        x, se, sol = fit(y, off)
        z = normal.ppf(0.975)
        records.append({"rep": rep, "model": label, "lnD": x[0], "lnk": x[1], "se_lnD": se[0], "se_lnk": se[1],
                        "b_hat": x[2] if off else np.nan, "cover_D": abs(x[0] - np.log(P.D)) <= z * se[0],
                        "cover_k": abs(x[1] - np.log(P.k)) <= z * se[1], "status": sol.status})
print(f"transient fits: {time.time() - t0:.0f} s")
fits = pd.DataFrame(records)
fits.to_csv(paths.DATA / "transient_fits.csv", index=False)
rows = []
for label, d in fits.groupby("model"):
    Dh, kh = np.exp(d.lnD), np.exp(d.lnk)
    row = {"model": label, "D mean": Dh.mean(), "D rel bias": Dh.mean() / P.D - 1, "D rel sd": Dh.std(ddof=1) / P.D,
           "k mean": kh.mean(), "k rel bias": kh.mean() / P.k - 1, "k rel sd": kh.std(ddof=1) / P.k,
           "b_hat mean": d.b_hat.mean(), "converged": (d.status > 0).mean()}
    for par in ("D", "k"):
        c = int(d[f"cover_{par}"].sum()); lo, hi = uncertainty.wilson_interval(c, len(d))
        row[f"coverage {par}"] = c / len(d); row[f"coverage {par} CI"] = f"[{lo:.3f}, {hi:.3f}]"
    rows.append(row)
fit_summary = pd.DataFrame(rows)
fit_summary.to_csv(paths.TABLES / "transient_fit_summary.csv", index=False)
print(fit_summary.T.to_string())
SUMMARY["transient_fit"] = {"sigma": sigma_t, "b": b_t, "R": R_T, "n_obs": int(T_OBS.size), "peak_true": float(y_true.max()),
                            "numerical_bias": num_bias, "summary": fit_summary.to_dict(orient="records")}
np.savez_compressed(paths.DATA / "transient_fit_data.npz", t_obs=T_OBS, y_true=y_true,
                    example_y=y_true + b_t + sigma_t * child_rngs("example_noise", 1)[0].standard_normal(T_OBS.size))

peak of true exit curve 0.1303 mol m^-3; sigma / peak = 0.015


noise-free fit (pure discretisation effect): relative errors {'D_rel': '1.45e-04', 'k_rel': '-2.52e-04'}


transient fits: 52 s
                            0                    1
model            M1 no offset  M2 offset estimated
D mean               0.010217                 0.01
D rel bias           0.021749             0.000037
D rel sd             0.016474             0.016775
k mean               0.096748             0.100021
k rel bias          -0.032519             0.000213
k rel sd             0.008307             0.009479
b_hat mean                NaN             0.002018
converged                 1.0                  1.0
coverage D               0.89                 0.93
coverage D CI  [0.839, 0.926]       [0.886, 0.958]
coverage k               0.11                 0.96
coverage k CI  [0.074, 0.161]       [0.923, 0.980]


## 7. Error budget at the baseline

The four error types are not interchangeable. The table places them side by side for each quantity of interest:
discretisation error of the production solver (notebook 02), RQMC and MC standard errors of the estimated mean at
$n=4096$, and the parametric standard deviation (the quantity of physical interest).

In [12]:
prod_acc = pd.read_csv(paths.TABLES / "production_accuracy.csv").set_index("QoI")
rows = []
for j, name in enumerate(QN):
    rows.append({"QoI": name, "unit": qoi.QOI_UNITS[name], "parametric sd": np.sqrt(ref_var[j]),
                 "discretisation error": prod_acc.loc[name, "abs error"],
                 "MC SE (n=4096)": replicates["mc"]["mean"][:, -1, j].std(ddof=1),
                 "RQMC SE (n=4096)": replicates["rqmc"]["mean"][:, -1, j].std(ddof=1)})
budget = pd.DataFrame(rows)
budget["disc / parametric sd"] = budget["discretisation error"] / budget["parametric sd"]
budget.to_csv(paths.TABLES / "error_budget.csv", index=False)
print(budget.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
SUMMARY["error_budget"] = budget.to_dict(orient="records")
reporting.save_json(SUMMARY, "nb03_summary.json")
print("saved nb03 summary")

     QoI       unit  parametric sd  discretisation error  MC SE (n=4096)  RQMC SE (n=4096)  disc / parametric sd
   C_out   mol m^-3         0.0761              1.16e-05         0.00117          5.56e-06              0.000153
       X          -         0.0732              1.16e-05         0.00119          4.71e-06              0.000159
    peak   mol m^-3         0.0296              6.34e-05        0.000427          2.89e-06               0.00214
  t_mean          s          0.745               0.00061          0.0129           6.4e-05              0.000819
exposure mol s m^-3          0.191              2.91e-05         0.00293          1.39e-05              0.000152
saved nb03 summary
